# 05 - Classification Metrics & Decision Thresholds

Notebook 01 printed a `classification_report` without saying what to do with it.
Accuracy is the wrong metric more often than beginners expect, and the default
0.5 decision threshold is a choice, not a law.

By the end you will be able to:
1. Explain why accuracy lies on imbalanced data.
2. Read a confusion matrix and choose between precision and recall.
3. Move the decision threshold on purpose.
4. Pick between ROC-AUC and Average Precision.
5. Handle class imbalance with `class_weight`.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
np.random.seed(42)

heart = pd.read_csv("../data/heart-disease.csv")
X = heart.drop("target", axis=1)
y = heart["target"]
print(X.shape, y.value_counts().to_dict())

## 1. The accuracy trap

Build a deliberately imbalanced problem: 97% negative, 3% positive - fraud, rare disease,
machine failure. All the interesting problems look like this.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

X_imb, y_imb = make_classification(
    n_samples=5000, n_features=20, n_informative=4,
    weights=[0.97, 0.03], random_state=42)

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

print("class balance:", np.bincount(y_imb), f"({y_imb.mean():.1%} positive)")

lazy = DummyClassifier(strategy="most_frequent").fit(Xi_tr, yi_tr)
print(f"\nA model that always predicts 'no': accuracy = {lazy.score(Xi_te, yi_te):.3f}")
print("It catches exactly zero positives. Accuracy is useless here.")

## 2. The confusion matrix is the source of truth

Every other classification metric is a ratio computed from these four numbers.

|  | predicted 0 | predicted 1 |
|---|---|---|
| **actual 0** | TN | FP (false alarm) |
| **actual 1** | FN (missed case) | TP |

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

clf = LogisticRegression(max_iter=2000).fit(Xi_tr, yi_tr)
y_pred = clf.predict(Xi_te)

cm = confusion_matrix(yi_te, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"true negatives  {tn:5d}   false positives {fp:5d}  (false alarms)")
print(f"false negatives {fn:5d}   true positives  {tp:5d}  (caught)")

ConfusionMatrixDisplay(cm, display_labels=["no", "yes"]).plot(cmap="Blues", colorbar=False)
plt.title("Confusion matrix")
plt.show()

In [ ]:
print(classification_report(yi_te, y_pred, target_names=["no", "yes"], digits=3, zero_division=0))

### What each row of that report means

- **precision** = TP / (TP + FP) - *of everything I flagged, how much was real?*
  Optimise this when a false alarm is expensive (spam filter deleting a real email).
- **recall** = TP / (TP + FN) - *of everything real, how much did I catch?*
  Optimise this when a miss is expensive (cancer screening, fraud detection).
- **f1** - the harmonic mean of the two. A single number when both matter.
- **support** - how many real examples of that class exist. Always check it: a precision
  of 1.00 on a support of 3 means nothing.

**You cannot maximise both.** Raise recall and you flag more things, so precision falls.
The business decides which error is worse, not the model.

## 3. Moving the threshold

`.predict()` silently applies `probability >= 0.5`. That 0.5 is a default, not a truth.

In [ ]:
probs = clf.predict_proba(Xi_te)[:, 1]   # probability of class 1

from sklearn.metrics import precision_score, recall_score, f1_score

rows = []
for t in [0.10, 0.20, 0.30, 0.50, 0.70, 0.90]:
    pred_t = (probs >= t).astype(int)
    rows.append({
        "threshold": t,
        "flagged": int(pred_t.sum()),
        "precision": precision_score(yi_te, pred_t, zero_division=0),
        "recall": recall_score(yi_te, pred_t),
        "f1": f1_score(yi_te, pred_t),
    })
pd.DataFrame(rows).round(3)

In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(yi_te, probs)

plt.figure(figsize=(9, 4.5))
plt.plot(thr, prec[:-1], label="precision", color="#4C78A8")
plt.plot(thr, rec[:-1], label="recall", color="#F58518")
plt.axvline(0.5, ls="--", c="grey", label="default threshold 0.5")
plt.xlabel("decision threshold"); plt.ylabel("score")
plt.title("Lower the threshold: recall rises, precision falls")
plt.legend(); plt.grid(alpha=.3)
plt.show()

> **Practical use.** "We must catch at least 80% of fraud" is a recall target. Read the
> threshold off this plot that delivers it, then check what precision you have to live
> with. That is an engineering conversation, not a modelling one.

## 4. ROC-AUC vs Average Precision

Both summarise a model across *all* thresholds, so you can compare models before
committing to one.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score

fpr, tpr, _ = roc_curve(yi_te, probs)
auc = roc_auc_score(yi_te, probs)
ap = average_precision_score(yi_te, probs)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(fpr, tpr, color="#4C78A8", lw=2, label=f"ROC-AUC = {auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", c="grey", label="random guessing")
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate")
axes[0].set_title("ROC curve"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(rec, prec, color="#F58518", lw=2, label=f"Avg precision = {ap:.3f}")
axes[1].axhline(yi_te.mean(), ls="--", c="grey", label=f"baseline = {yi_te.mean():.3f}")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title("Precision-Recall curve"); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.show()

**Which one?**

- **ROC-AUC** - use when the classes are roughly balanced. It is optimistic on rare
  positives, because a huge TN count keeps the false positive rate low no matter what.
- **Average Precision (PR-AUC)** - use when positives are rare. Its baseline is the
  positive rate itself, so it is honest about hard problems.

Note above how the ROC-AUC looks excellent while average precision is far more sober.
Same model, same predictions.

## 5. Telling the model the classes are imbalanced

In [ ]:
plain = LogisticRegression(max_iter=2000).fit(Xi_tr, yi_tr)
weighted = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xi_tr, yi_tr)

for name, m in [("default", plain), ("class_weight='balanced'", weighted)]:
    p = m.predict(Xi_te)
    print(f"{name:26} | precision {precision_score(yi_te, p, zero_division=0):.3f} "
          f"| recall {recall_score(yi_te, p):.3f} | f1 {f1_score(yi_te, p):.3f}")

`class_weight="balanced"` makes each mistake on the rare class count more during training.
Recall goes up, precision goes down - the same trade as moving the threshold, applied at
fit time instead of predict time.

---
## Exercises

1. Change `weights=[0.97, 0.03]` to `[0.5, 0.5]`. Do ROC-AUC and average precision still
   disagree? Explain what changed.
2. Find the threshold that gives at least **90% recall**. What precision do you get, and
   how many false alarms per 1000 cases is that?
3. Run the whole notebook on `heart-disease.csv` instead. Is it imbalanced? Which metric
   would you report to a doctor, and why?
4. Compare `class_weight="balanced"` against simply setting the threshold to 0.2. Do they
   land in the same place? What is different about them in practice?
5. **Stretch:** write `best_threshold(y_true, probs, min_recall=0.8)` that returns the
   highest-precision threshold meeting a recall floor. Use it to pick an operating point.